<a href="https://colab.research.google.com/github/padmasundarg95/Project-NEXUS-Agent/blob/main/Phase_2_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
#Project NEXUS - Phase 2 | RAG Engine

#---------------------------------------------Step 1 - Installing required libraries-------------------------------------------------#

#Installing langchain framework (provides document container), langchain is the orchestration framework that is used, this acts as a manager telling every section what to do
#langchain community - langchain to talk to chroma db,
#langchain-google-genai - connects langchain to gemini (convert text to vectors)
#chromadb - Vector db that stores the slices
#groq - connects to groqs api so that llama can generate the final answer
!pip install -q -U langchain langchain-community langchain-google-genai chromadb groq

#---------------------------------------------Step 2 - Document Container - Raw data & Meta data-----------------------------------------#

import site
from importlib import reload
reload(site)
from langchain_core.documents import Document

nexus_docs = []
if 'blogs_content' in globals() and blogs_content:
    for blog in blogs_content:
        for slice_text in blog['chunks']:
            metadata = {"title": blog['Blog title'], "link": blog['Blog link']} #metadata is the source of reference for the contents - blog titles and links
            nexus_docs.append(Document(page_content=slice_text, metadata=metadata)) #page content is the raw data that is sent to embedding model for conversion to vectors
    print(f"Total Searchable Slices: {len(nexus_docs)}")
else:
    print("'blogs_content' is empty. Run the Scraper cell first.")

#--------------------------------------------Step 3 - Embedding the chunks and storing them into DB---------------------------------------#
from google.colab import drive, userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
import time

drive.mount('/content/drive') #Mounting connects the google drive to colab so that chroma db doesn't get deleted after every session restart
target_dir = "/content/drive/MyDrive/nexus_rag/chroma_db"

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    google_api_key=userdata.get('GOOGLE_API_KEY'),
    task_type="retrieval_document"
)

vector_db  = Chroma(persist_directory=target_dir, embedding_function=embeddings) #persist dictionary tells chroma db to save the slices directly into drive and not on colab (temp machine)
all_docs   = nexus_docs
batch_size = 20
total      = len(all_docs)
max_retries = 5

i = vector_db._collection.count()
print(f"Resuming from slice {i}/{total}")

while i < total:
    batch   = all_docs[i : i + batch_size]
    retries = 0
    success = False
    while retries < max_retries: #exponential backoff logic to handle 429 rate limit error
        try:
            vector_db.add_documents(batch)
            i = vector_db._collection.count()
            print(f"[{i}/{total}] slices locked in.")
            time.sleep(5)
            success = True
            break
        except Exception as e:
            if "429" in str(e):
                wait_time = 60 * (2 ** retries)
                print(f"⏳ Quota hit. Retry {retries+1}/{max_retries} in {wait_time}s...")
                time.sleep(wait_time)
                retries += 1
            else:
                print(f"Unexpected error: {e}")
                break
    if not success:
        print(f"Stopped at {i}. Re-run tomorrow to resume.")
        break

print(f"🏁 Done: {vector_db._collection.count()}/{total} slices stored.")

#--------------------------------Step 4 - Query Engine - Embeds the user question into a vector,
                                #-------------retrieves top 5 matching chunks from ChromaDB,
                              #--------------------and generates the final answer using Groq/Llama-------------------#
import chromadb
from google import genai
from groq import Groq #Groq hosts Llama
from google.colab import userdata

CHROMA_PATH     = "/content/drive/MyDrive/nexus_rag/chroma_db" #drive where chroma db stores the slices
COLLECTION_NAME = "langchain" #collection name where the slices are saved in drive
GROQ_MODEL      = "llama-3.3-70b-versatile"
TOP_K           = 5 #searches for top 5 similarity chunks

gemini_client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY")) #connect to gemini api using the secret key, used to convert qns into vectors
groq_client   = Groq(api_key=userdata.get("GROQ_API_KEY")) #connect to groq api using groq secret key, used to generate final answer using llama
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH) #persistentclient refers to reading from the saved file and not from an empty fresh db
collection    = chroma_client.get_collection(name=COLLECTION_NAME) #goes inside chroma db and picks the specific collection name langchain where 3331 slices are saved

print(f"Collection '{COLLECTION_NAME}' loaded — {collection.count()} slices")

# defines a function, accepts text as input, returns a list of numbers
def embed_query(text: str) -> list[float]: #converts user's question into vectors to compare it against blog vectors
    result = gemini_client.models.embed_content( #calls gemini api
        model="models/gemini-embedding-2", #use this specific model
        contents=text, #sends question text to gemini
    )
    return result.embeddings[0].values  # from the response, extract the first embedding's numbers and return it

 #take the qn vector, searches chroma db, returns top 5 closest blog chunks with their metadata
def retrieve(query: str, top_k: int = TOP_K) -> list[dict]:# accepts a question and how many chunks to fetch (default 5)
    query_embedding = embed_query(query) # first convert the question into a vector
    results = collection.query( #search chroma db
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"], #returns the text, title/link, similarity score
    )
    chunks = [] #empty basket to collect results
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ): #loop through each of the 5 results together
        chunks.append({"text": doc, "metadata": meta, "distance": dist}) #pack each result into a dict and add to basket
    return chunks #return all 5 packed chunks

def generate(query: str, chunks: list[dict]) -> str: # accepts the question and the 5 retrieved chunks, returns a text answer
    context = "\n\n---\n\n".join(     # join all chunks into one block of text, separated by ---
        f"[{c['metadata']['title']} | {c['metadata']['link']}]\n{c['text']}"  # for each chunk, show title | link on top, then the text below
        for c in chunks #this is done for all 5 chunks
    )
    response = groq_client.chat.completions.create( # call Groq API to generate an answer
        model=GROQ_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a support agent for the blogs portfolio by PadmaSundar." #assigning LLM the role - persona prompting
                    "Answer using only the blog excerpts below. " #output schema enforcement
                    "If the context doesn't have enough info, say that honestly.\n\n" #preventing hallucination
                    f"BLOG EXCERPTS:\n{context}" #paste all 5 chunks for Llama to read
                )
            },
            {"role": "user", "content": query}, #Question from the user
        ],
        temperature=0.2, #temperature is how creative or strict AI is when answering a question (0.2  → mostly factual, slight natural flow  ← you are here)
        max_tokens=1024, #tokens refer to the length of the answers - 1024 tokens = ~750 words , apprx 1 token = 0.75 words
    )
    return response.choices[0].message.content #extract the answer from groq's response and return it

def rag_query(question: str, top_k: int = TOP_K) -> str: #master function - accepts a qn and returns the ans as string
    print(f"\n Query: {question}") #print the question
    chunks = retrieve(question, top_k=top_k) #fetch the 5 most relevant chunks
    print(f"\n Retrieved {len(chunks)} chunks:")
    for i, c in enumerate(chunks, 1):
        print(f"  [{i}] dist={c['distance']:.4f} | {c['metadata']['title']}")  #prnt each chunk's rank(lower the distance better the rank and higher the match), similarity score and title
    answer = generate(question, chunks) #send chunks + question to Llama
    return answer #returns the answer

#------------------------------------------------------------Step 5: Ask a question------------------------------------------------------#
rag_query("Tell me about this writer")

Collection 'langchain' loaded — 3331 slices

 Query: Tell me about this writer

 Retrieved 5 chunks:
  [1] dist=0.6722 | the spies of God - part1
  [2] dist=0.6731 | A documentary that got me smizing🧡
  [3] dist=0.6938 | 8 am Metro - A movie that felt like a 'Jigiri' poem
  [4] dist=0.7044 | Stories from Mumbai [Chapter 2: Off on a journey with Sudha Murthy's Common Yet Uncommon]
  [5] dist=0.7266 | Ramum Anandhiyum pine ente manassu thotta mattu chilarum...


"Based on the provided blog excerpts, it appears that the writer, PadmaSundar, is a reflective and appreciative individual who enjoys exploring various forms of media, such as books, documentaries, and movies. They seem to have a deep appreciation for authors and their writing styles, often expressing admiration for the way certain writers can evoke emotions and create relatable experiences.\n\nThe writer also appears to be a fan of specific authors, such as Sudha Murthy, and is enthusiastic about discovering new stories and learning more about the people they admire. They have a poetic and introspective tone in their writing, often using metaphors and philosophical questions to express their thoughts and feelings.\n\nHowever, the context doesn't provide enough information about the writer's personal life, background, or professional experience."

In [1]:
#To Run the RAG Engine every time, follow the below steps instead of running the entire code.
#Connect Google Drive to Colab so the query engine can read the saved chroma db and find all 3331 slices
# 1. Install
#!pip install -q chromadb groq google-genai

# 2. Mount Drive
#from google.colab import drive
#drive.mount('/content/drive')

# 3. Run Step 4, 5 from above - ChromaDB reads from Drive.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t